# Notebook 15 — Verified tuning-source exclusion and paired sensitivity

**Research status: NOT EXECUTED.** Computes the pre-specified exclusion of the **12 provenance-verified NB06c tuning sources** from the frozen N07/N12 120-source experiment, without any new generation or re-training.

**Privacy caution:** ArcFace/AuraFace cosine is **not** calibrated re-identification risk. Outputs are aggregate-only; raw original, generated images, individual biometric templates, or per-source data must stay on the private Drive.

**Exclusion evidence:** NB06c guidance run `20260817T211622420456Z_flowedit-nb06c-guidance-below-flowedit-low_nb06c-guidance-12src-12aef418/selected_sources.csv`; N07 `selected_sources.csv`. For all 12 IDs, `prepared_source_sha256` matches exactly (independently checked against Drive). The notebook repeats the provenance assertion in Colab.

**Scientific comparison:** the overall vs 108-source subset is a sensitivity analysis; paired g5.5-minus-g4.5 source-level comparisons are reported separately by seed and across seeds. No p-value or privacy-threshold claim is inferred from aggregate CIs.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import json
import hashlib

# Notebook 06c: older thesis_runs directory
NB06 = Path(
    '/content/drive/MyDrive/thesis_runs/'
    '20260817T211622420456Z_flowedit-nb06c-guidance-below-flowedit-low_nb06c-guidance-12src-12aef418'
)

# Notebooks 07 and 12: newer runs directory
ROOT = Path('/content/drive/MyDrive/thesis_all/outputs/runs')

NB07 = ROOT / '20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'

NB12 = ROOT / 'nb12_v3_biometric_11faafd9beb1_20261013T211442695856Z_bcf37701'

# Verify input directories
for name, path in {
    'NB06': NB06,
    'NB07': NB07,
    'NB12': NB12,
}.items():
    assert path.is_dir(), f'{name} directory missing: {path}'
    print(f'{name}: OK')

# New output directory
out = ROOT / (
    'nb15_verified_tuning_exclusion_'
    + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
)

out.mkdir(parents=True, exist_ok=False)
print('OUTPUT:', out)


In [ ]:
# Provenance check: exact prepared image hashes, not merely matching source numbers.
ids_expected=['source_000','source_001','source_005','source_006','source_010','source_011',
              'source_015','source_016','source_020','source_021','source_025','source_026']
tune=pd.read_csv(NB06/'selected_sources.csv', dtype={'source_id':'string'})
selected=pd.read_csv(NB07/'selected_sources.csv', dtype={'source_id':'string'})
assert len(tune)==12 and tune.source_id.is_unique and set(tune.source_id)==set(ids_expected)
assert len(selected)==120 and selected.source_id.is_unique
cols=['source_id','prepared_source_sha256']
assert set(cols).issubset(tune.columns) and set(cols).issubset(selected.columns)
compare=tune[cols].merge(selected[cols],on='source_id',validate='one_to_one',suffixes=('_nb06','_nb07'))
assert len(compare)==12 and (compare.prepared_source_sha256_nb06==compare.prepared_source_sha256_nb07).all(), 'PROVENANCE MISMATCH: STOP'
print('12/12 image SHA256 provenance matches. Exact tuning IDs:', ids_expected)


In [ ]:
# Reconcile original utility and corrected privacy row-for-row.
keys=['source_id','condition','seed']
n07=pd.read_csv(NB07/'raw_results.csv',dtype={'source_id':'string','condition':'string'})
n12=pd.read_csv(NB12/'raw_corrected_privacy.csv',dtype={'source_id':'string','condition':'string'})
for label,frame in [('N07',n07),('N12',n12)]:
    assert len(frame)==600 and not frame.duplicated(keys).any(),f'{label} invalid keys'
assert set(map(tuple,n07[keys].itertuples(index=False,name=None)))==set(map(tuple,n12[keys].itertuples(index=False,name=None)))
assert n07.source_id.nunique()==n12.source_id.nunique()==120
assert (n07.emotion_status.astype(str)=='success').all()
assert (n12.pair_valid_arcface.astype(str).str.lower()=='true').all()
assert (n12.pair_valid_auraface.astype(str).str.lower()=='true').all()
keep=['source_id','condition','seed','emotion_preserved']
df=n12[keys+['arcface_cosine','auraface_cosine']].merge(n07[keep],on=keys,validate='one_to_one')
df['emotion_preserved']=df.emotion_preserved.astype(str).str.lower().map({'true':1.,'false':0.})
assert df[['arcface_cosine','auraface_cosine','emotion_preserved']].notna().all().all()
assert (df.groupby('source_id').size()==5).all()
remaining=df[~df.source_id.isin(ids_expected)].copy()
assert len(remaining)==540 and remaining.source_id.nunique()==108
print('600 complete pairs, 540 after removal of 12 tuning sources.')


In [ ]:
# Aggregate report + cluster bootstrap by source image (prevents pseudo-replication).
METRICS=['arcface_cosine','auraface_cosine','emotion_preserved']
RNG=np.random.default_rng(15015)
def bootstrap_by_source(frame,col,draws=2000):
    grouped=[g[col].to_numpy(dtype=float) for _,g in frame.groupby('source_id')]
    n=len(grouped)
    assert n>1
    values=np.empty(draws)
    for j in range(draws):
        selected_ids=RNG.integers(0,n,n)
        values[j]=np.concatenate([grouped[i] for i in selected_ids]).mean()
    return np.quantile(values,[0.025,0.975]).tolist()
rows=[]
for subset_name,frame in [('ALL_120',df),('EXCLUDE_12_108',remaining)]:
    for condition,g in frame.groupby('condition',sort=True):
        for metric in METRICS:
            lo,hi=bootstrap_by_source(g,metric)
            rows.append(dict(subset=subset_name,condition=condition,metric=metric,
                n_pairs=len(g),n_sources=g.source_id.nunique(),mean=float(g[metric].mean()),ci95_low=lo,ci95_high=hi))
summary=pd.DataFrame(rows).sort_values(['subset','condition','metric'])
summary.to_csv(out/'tuning_exclusion_aggregate.csv',index=False)
display(summary.round(5))


In [ ]:
# Matched comparison: same original and seed, guidance g5.5 versus g4.5.
conditions=set(df.condition.unique())
g4='flowedit_confirmatory_g4p5';g5='flowedit_g5p5'
assert {g4,g5,'weak_edit_control'}==conditions, conditions
results=[]
for subset_name,frame in [('ALL_120',df),('EXCLUDE_12_108',remaining)]:
    pairs=frame[frame.condition.isin([g4,g5])].pivot(index=['source_id','seed'],columns='condition',values=METRICS)
    assert pairs.notna().all().all(), 'Unmatched paired conditions'
    assert len(pairs)==(240 if subset_name=='ALL_120' else 216)
    for metric in METRICS:
        diff=(pairs[(metric,g5)]-pairs[(metric,g4)]).rename('delta').reset_index()
        for seed_name,part in [('BOTH_SEEDS',diff),*[(str(seed),diff[diff.seed==seed]) for seed in sorted(diff.seed.unique())]]:
            lo,hi=bootstrap_by_source(part,'delta')
            results.append(dict(subset=subset_name,seed=seed_name,metric=metric,n_pairs=len(part),n_sources=part.source_id.nunique(),
                 mean_g5_minus_g4=float(part.delta.mean()),cluster_CI95_low=lo,cluster_CI95_high=hi))
paired=pd.DataFrame(results)
paired.to_csv(out/'paired_guidance_delta_aggregate.csv',index=False)
display(paired.round(5))


In [ ]:
# Publish only aggregate analyses, never raw per-image frames.
report={'method':'Verified NB06c tuning exclusion for frozen N07-N12',
  'tuning_source_ids':ids_expected,  # private output: source-level provenance metadata
  'provenance':'12/12 matching prepared_source_sha256 values',
  'full_pairs':600,'full_sources':120,'after_exclusion_pairs':540,'after_exclusion_sources':108,
  'privacy_calibration':'NOT CALIBRATED',
  'limitations':['Potential original-identity duplication not independently verified',
                 'No independent second QC review','Cosine similarity is not probability of reidentification'],
  'files':['tuning_exclusion_aggregate.csv','paired_guidance_delta_aggregate.csv']}
(out/'evidence_status_private.json').write_text(json.dumps(report,indent=2))
print('Completed; aggregate results saved in',out)
print('Keep evidence_status_private.json in private Drive; do not commit it to public GitHub.')
